# Puzzle

### This Week’s Fiddler
Congratulations to Fiddler Nation for making it to the semifinals of the World Cup! All four teams that made it this far are equally matched in that they each possess the same total amount of “energy.” In advance of each semifinal game, teams must independently decide how much of their energy to allocate to the match; all remaining energy goes toward the finals. The team that spends more energy in any given game will win. The semifinals and finals occur so close in time that teams can’t recuperate any of their energy in between.

You’ve heard that the managers for the other three teams are abysmal and have no idea how to allocate their teams’ energy. Each of the other managers will independently pick a random percentage between 0 and 100 and allocate that portion of their team’s energy to the semifinal game; the rest of that team’s energy will go toward the final.

Since you’re the cleverest manager of the bunch, you can choose an optimal strategy that will maximize Fiddler Nation’s probability of winning the World Cup. What is this optimal probability?

### This Week’s Extra Credit
As it turns out, I spoke too soon. Fiddler Nation has made it to the quarterfinals of the World Cup rather than the semifinals. My mistake. As before, teams must allocate the same total amount of energy across up to three matches.

The managers for the other seven teams remain abysmal. Each manager will independently pick a random percentage between 0 and 100 and allocate that amount of their team’s energy to the quarterfinal. If they win, they will allocate a random amount of their remaining energy to the semifinal. And if they win that, the rest of their team’s energy will go toward the final.

Fiddler Nation’s strategy must be drawn up in advance, with no specific knowledge of the other teams’ strategies beyond what I have already shared.

That said, as the cleverest manager of the bunch you can once again choose an optimal strategy that will maximize Fiddler Nation’s probability of winning the World Cup. What is this optimal probability?

# Fiddler solution

Let x be the fraction of energy you allocate to the first game, and let y,z,w be the other team's choices.

Probability of winning the first game = P(x > y) = x.

In the second game, your energy will be (1-x) and your probability of winning is the probability that your opponent has less energy than that.

Opponent's energy is 1 - max(z,w).

P(You win) 

= P(1- max(z,w) < 1-x )

= P(max(z,w) > x)

= 1 - P(max(z,w) <= x)

= 1 - P(z and w are both less than x)

= 1 - x^2

So, overall probability of winning the world cup = x (1 - x^2) = x - x^3

To maximize this, we calculate the derivative = 1 - 3x^2, set it to 0.

So, $x = 1/\sqrt{3}$

And $P = 2/3\sqrt{3}$ = 0.38490017945975050967276585366797

That's a good bit higher than the 0.25 from a uniform random approach.

# Extra Credit

We have more teams and more variables.

Let's use x and y for our 2 choices (qtr-final, semi-final), and a,b,c, etc for choices by other teams.

### Game 1

So, our energy in the 3 matches is x, (1-x)y, (1-x)(1-y).

P(winning first match) = x

### Game 2 

P(winning second match) = P( ((1-max(a,b))*c) < k ), where k = (1-x)y

((1-max(a,b))*c) < k

=> 1-max(a,b) < k/c

=> max(a,b) > 1 - k/c = q

P( max(a,b) > q)

= 1 - P( max(a,b) < q)

= 1 - P( both a and b are < q)

= 1 - q^2

= 1 - (1-k/c)^2

= 2k/c + k^2/c^2

Averaging over all values of c uniformly distributed, gives an integral that will diverge. ( ln(c) and 1/c evaluated at 0 )

So, dividing by c was not a good approach.

### Coding

Code Run elsewhere

In [2]:

from random import random as rnd

def one_tournament(x,y):
    e1, e2, e3 = x, (1-x)*y, (1-x)*(1-y)

    # q1
    a = rnd()
    if (e1 < a):
        return 0

    # q2
    b = rnd()
    c = rnd()
    bc = max(b,c)
    d = rnd()
    bcd = (1-bc) * d

    # s1
    if (e2 < bcd) :
        return 0

    # q3
    f = rnd()
    g = rnd()
    fg = max(f,g)
    h = rnd()
    fgh = (1-fg) * h

    # q4
    i = rnd()
    j = rnd()
    ij = max(i,j)
    k = rnd()
    ijk = (1-ij) * k

    # s2 :
    s2 = 0
    if (fgh > ijk):
        s2 = (1-fg)*(1-h)
    else:
        s2 = (1-ij)*(1-k)

    # final
    if (s2 > e3):
        return 0
    else:
        return 1

    
#for i in range(10):
#    print(one_tournament(.2,.3))

def get_pxy(x,y,N_TRIALS=100):
    wins=0
    for i in range(N_TRIALS):
        wins += one_tournament(x,y)
    p = 1.0*wins/N_TRIALS
    return p


def find_best_pxy(N_GRID=100, N_TRIALS=1000, SMOOTH_R=5, xmin=0.0,xmax=1.0,ymin=0.0,ymax=1.0):
    pxy = [ [ 0 for _ in range(N_GRID+1)] for _ in range(N_GRID+1)]
    for i in range(N_GRID+1):
        x = xmin + (xmax-xmin)*(i/N_GRID)
        for j in range(N_GRID+1):
            y = ymin + (ymax-ymin)*(j/N_GRID)
            p = get_pxy(x,y,N_TRIALS)
            pxy[i][j] = p
    smooth_pxy = [ [ 0 for _ in range(N_GRID+1)] for _ in range(N_GRID+1)]
    for i in range(N_GRID+1):
        for j in range(N_GRID+1):
            local_sum, lcnt = 0,0
            for di in range(-SMOOTH_R, SMOOTH_R+1):
                ni = i+di
                if (0 <= ni <= N_GRID):
                    for dj in range(-SMOOTH_R, SMOOTH_R+1):
                        nj = j+dj
                        if (0 <= nj <= N_GRID):
                            local_sum += pxy[i][j]
                            lcnt += 1
            smooth_pxy[i][j] = 1.0*local_sum/lcnt
    best_p,best_x,best_y = -1,0,0
    for i in range(1,N_GRID):
        for j in range(1,N_GRID):
            p = pxy[i][j]
            if p > best_p:
                best_p = p
                x = xmin + (xmax-xmin)*(i/N_GRID)
                y = ymin + (ymax-ymin)*(j/N_GRID)
                best_x = x
                best_y = y

    print(f"{N_GRID=}, {N_TRIALS=} {SMOOTH_R=}: {best_p=}, {best_x=}, {best_y=}")
    return best_p, best_x, best_y


# print(find_best_pxy(N_GRID=10))
# print(find_best_pxy(N_GRID=100))
# print(find_best_pxy(N_GRID=200))
# print(find_best_pxy(N_GRID=300))
# print(find_best_pxy(N_GRID=500))
# print(find_best_pxy(N_GRID=600))
# print(find_best_pxy(N_GRID=1000))
#
# (0.298, 0.5, 0.6)
# (0.316, 0.55, 0.42)
# (0.336, 0.54, 0.355)
# (0.329, 0.5233333333333333, 0.46)
# (0.331, 0.508, 0.518)
# (0.332, 0.4816666666666667, 0.43833333333333335)



# Those runs seem to indicate that the best x may be around 0.3 and best y may be around 0.5
# So, let's do a final run

# find_best_pxy(N_GRID=400, N_TRIALS=10000, xmin=0.2, xmax=0.4, ymin=0.4, ymax=0.6)
# N_GRID=400, N_TRIALS=10000: 0.2767, 0.3995, 0.4815

# Wait, I have been misreading the earlier printouts.
#find_best_pxy(N_GRID=100, N_TRIALS=10000, SMOOTH_R=10, xmin=0.4, xmax=0.6, ymin=0.4, ymax=0.6)
# N_GRID=100, N_TRIALS=10000 SMOOTH_R=10: best_p=0.2952, best_x=0.472, best_y=0.464

# find_best_pxy(N_GRID=100, N_TRIALS=10000, SMOOTH_R=10, xmin=0.4, xmax=0.6, ymin=0.3, ymax=0.7)
# N_GRID=100, N_TRIALS=10000 SMOOTH_R=10: best_p=0.297, best_x=0.524, best_y=0.45999999999999996
# N_GRID=100, N_TRIALS=10000 SMOOTH_R=10: best_p=0.295, best_x=0.506, best_y=0.524
# N_GRID=100, N_TRIALS=10000 SMOOTH_R=10: best_p=0.2943, best_x=0.518, best_y=0.428

# find_best_pxy(N_GRID=100, N_TRIALS=100000, SMOOTH_R=10, xmin=0.45, xmax=0.55, ymin=0.35, ymax=0.6)
# N_GRID=100, N_TRIALS=100000 SMOOTH_R=10: best_p=0.28597, best_x=0.53, best_y=0.495
# N_GRID=100, N_TRIALS=100000 SMOOTH_R=10: best_p=0.28552, best_x=0.51, best_y=0.45249999999999996
 
find_best_pxy(N_GRID=25, N_TRIALS=100000, SMOOTH_R=10, xmin=0.45, xmax=0.55, ymin=0.4, ymax=0.55)
# N_GRID=25, N_TRIALS=100000 SMOOTH_R=10: best_p=0.28449, best_x=0.526, best_y=0.46
# N_GRID=25, N_TRIALS=100000 SMOOTH_R=10: best_p=0.28526, best_x=0.514, best_y=0.49000000000000005

N_GRID=25, N_TRIALS=100000 SMOOTH_R=10: best_p=0.2846, best_x=0.53, best_y=0.47200000000000003


(0.2846, 0.53, 0.47200000000000003)

0.285 or thereabouts.